# 第12回 関数データ解析

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：何百本もの曲線の違いは、いくつの数で言い表せるか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "tsibbledata", "feasts", "fda"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(tsibbledata)
library(feasts)
library(ggtime)   # ACFの図（autoplot）
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)
invisible(Sys.setlocale("LC_TIME", "C"))   # 図の月の名前を英語にする

`fda`パッケージは、第2回と同じく`library()`で読み込まず、`fda::`を付けて呼びます。

### A-1. 曲線を何本も作る

ここまでの回は、1本の長い系列を扱いました。今日は、短い曲線が何本もあるデータを扱います。
1本の曲線を、1つの観測とみます。このようなデータを、関数データ（functional data）といいます。

人工の曲線を200本作ります。1本は「1日」で、30分ごとに48点あります。作り方は次のとおりです。

$$
y_{ij} = \mu(t_j) + z_{i1}\,\xi_1(t_j) + z_{i2}\,\xi_2(t_j) + \varepsilon_{ij}
$$

- $\mu(t)$：どの曲線にも共通の形（平均関数）
- $\xi_1(t)$：1日全体を上げ下げする形。$\xi_2(t)$：午前を上げ、午後を下げる形
- $z_{i1}$、$z_{i2}$：曲線ごとに違う数（分散は6と2）。曲線ごとの違いは、この2つの数だけで決まる
- $\varepsilon_{ij}$：測るときのノイズ（標準偏差0.2）

In [ ]:
# 1日の中の時刻（時）。30分ごとの区間のまん中で、48点ある
tt <- seq(0.25, 23.75, by = 0.5)
# 曲線の行列（48行 × 本数）を、図のための縦長の表にする
long_curves <- function(M, idx = seq_len(ncol(M))) {
  data.frame(time = rep(tt, length(idx)), curve = rep(idx, each = length(tt)),
             value = as.numeric(M[, idx]))
}
# 曲線を重ねて描く
plot_curves <- function(M, idx = seq_len(ncol(M)), y_lab = "Value") {
  ggplot(long_curves(M, idx), aes(time, value, group = curve)) +
    geom_line(alpha = 0.3) +
    scale_x_continuous(breaks = seq(0, 24, 6)) +
    labs(x = "Time of day (hour)", y = y_lab)
}
# 誤差（第2回と同じ）
rmse <- function(estimate, truth) sqrt(mean((estimate - truth)^2))
mae  <- function(estimate, truth) mean(abs(estimate - truth))

In [ ]:
set.seed(1)
N <- 200                                        # 曲線の本数
mu_true <- 5 + sin(2 * pi * (tt - 9) / 24)      # 平均関数
xi1 <- rep(1 / sqrt(24), 48)                    # 1日全体を上げ下げする形
xi2 <- sin(2 * pi * tt / 24) / sqrt(12)         # 午前を上げ、午後を下げる形
nu_true <- c(6, 2)                              # 曲線ごとの数 z1, z2 の分散
z1 <- rnorm(N, sd = sqrt(nu_true[1]))
z2 <- rnorm(N, sd = sqrt(nu_true[2]))
X_true <- mu_true + outer(xi1, z1) + outer(xi2, z2)       # ノイズのない曲線（48行 × 200列）
Y_sim  <- X_true + matrix(rnorm(48 * N, sd = 0.2), 48)    # 観測（ノイズつき）
dim(Y_sim)
plot_curves(Y_sim)

### A-2. 点の列を、なめらかな関数に戻す（第2回の復習）

観測は48個の点です。その背後に、なめらかな関数 $x_i(t)$ があると考えます。

$$
y_{ij} = x_i(t_j) + \varepsilon_{ij}
$$

第2回と同じ方法で、$x_i(t)$ を推定します。

- 関数を、基底関数（Bスプライン）の重みつきの和で表す。今日は、基底関数を50個置く（30分に約1個）。
- 重みは、「観測点からのずれ ＋ $\lambda$ × 曲線の曲がりの大きさ」が最も小さくなるように決める。
- $\lambda$（粗さの罰則の強さ）は、GCVが最も小さくなる値を選ぶ。200本に同じ $\lambda$ を使い、GCVは200本の平均で比べる。

In [ ]:
# 基底関数（Bスプライン、30分に約1個）
basis <- fda::create.bspline.basis(c(0, 24), nbasis = 50)
# すべての曲線を、同じλでなめらかにする（Y は 48行 × 本数）
smooth_curves <- function(Y, lambda) {
  fda::smooth.basis(tt, Y, fda::fdPar(basis, 2, lambda))
}
# GCV（曲線についての平均）が最小になるλを選ぶ
choose_lambda <- function(Y, log10_lambda = seq(-5, 3, 0.5)) {
  gcv <- sapply(log10_lambda, function(l) mean(smooth_curves(Y, 10^l)$gcv))
  list(lambda = 10^log10_lambda[which.min(gcv)],
       table = data.frame(log10_lambda = log10_lambda, gcv = gcv))
}
# なめらかにした曲線の、48個の時刻での値（48行 × 本数）
curve_values <- function(fd) fda::eval.fd(tt, fd)
# 図：1本の曲線について、観測の点と、なめらかにした曲線（細かい刻みで描く）
plot_one <- function(Y, fd, i, y_lab = "Value") {
  fine <- seq(0.25, 23.75, by = 0.05)
  ggplot() +
    geom_point(data = data.frame(time = tt, value = Y[, i]), aes(time, value), colour = "grey50") +
    geom_line(data = data.frame(time = fine, value = as.numeric(fda::eval.fd(fine, fd[i]))),
              aes(time, value), colour = "red") +
    scale_x_continuous(breaks = seq(0, 24, 6)) +
    labs(x = "Time of day (hour)", y = y_lab)
}

In [ ]:
sel <- choose_lambda(Y_sim)
lambda <- sel$lambda
log10(lambda)
fd_sim <- smooth_curves(Y_sim, lambda)$fd     # 200本の関数
X_sim  <- curve_values(fd_sim)
plot_one(Y_sim, fd_sim, 1, "Value (curve 1)") +
  geom_line(data = data.frame(time = tt, value = X_true[, 1]), aes(time, value), linetype = "dashed")
# ノイズのない曲線とのずれ：なめらかにする前と後
c(observed = rmse(Y_sim, X_true), smoothed = rmse(X_sim, X_true))

- 灰色の点が観測、赤い線がなめらかにした曲線、破線がノイズのない曲線です。
- なめらかにすると、ノイズのない曲線とのずれが小さくなります。まわりの点を使って、ノイズをならすからです。

### A-3. 平均関数と共分散関数

第1回で、平均関数 $\mu_t = \mathrm{E}[x_t]$ と自己共分散関数 $\gamma(s, t) = \mathrm{Cov}(x_s, x_t)$ を定義しました。
そのときは、標本路が1本しかないので、時点ごとの平均を直接には推定できませんでした。

今日は曲線が200本あります。時刻ごとに、200本の平均と共分散をそのまま計算できます。

$$
\hat{\mu}(t) = \frac{1}{N} \sum_{i=1}^{N} x_i(t), \qquad
\hat{\gamma}(s, t) = \frac{1}{N - 1} \sum_{i=1}^{N} \{x_i(s) - \hat{\mu}(s)\}\{x_i(t) - \hat{\mu}(t)\}
$$

In [ ]:
mu_hat <- rowMeans(X_sim)                     # 時刻ごとに、200本を平均する
plot_curves(X_sim, 1:50) +
  geom_line(data = data.frame(time = tt, value = mu_hat, curve = 0), colour = "red", linewidth = 1.2) +
  geom_line(data = data.frame(time = tt, value = mu_true, curve = 0), linetype = "dashed")
rmse(mu_hat, mu_true)

赤い線が推定した平均関数、破線が本当の平均関数です。

共分散関数は、時刻 $s$ と時刻 $t$ の組ごとに値が1つあります。タイルの図で見ます。

In [ ]:
# 共分散関数（48 × 48 の行列）を、タイルの図にする
plot_cov <- function(G, fill_lab = "Covariance") {
  ggplot(data.frame(s = rep(tt, 48), t = rep(tt, each = 48), value = as.numeric(G)), aes(s, t, fill = value)) +
    geom_tile() +
    scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B") +
    scale_x_continuous(breaks = seq(0, 24, 6)) +
    scale_y_continuous(breaks = seq(0, 24, 6)) +
    coord_equal() +
    labs(x = "Time s (hour)", y = "Time t (hour)", fill = fill_lab)
}

In [ ]:
G_hat  <- cov(t(X_sim))                                              # 推定した共分散関数
G_true <- nu_true[1] * outer(xi1, xi1) + nu_true[2] * outer(xi2, xi2)   # 本当の共分散関数
plot_cov(G_hat)
# 3つの時刻（6時、12時、18時ごろ）での値：推定と本当の値
at <- c(12, 24, 36)
round(G_hat[at, at], 3)
round(G_true[at, at], 3)

- 対角線（$s = t$）は、その時刻での分散です。
- 対角線の外が0でないのは、1本の曲線の中で、違う時刻の値どうしが従属しているからです。
- 6時と18時の共分散は、6時どうしの分散より小さくなります。午前を上げて午後を下げる動き（$\xi_2$）が、この2つの時刻を逆向きに動かすからです。

### A-4. 関数主成分分析

主成分分析は、多くの変数を、少数の合成変数（主成分）にまとめる方法でした。
関数主成分分析（functional principal component analysis, FPCA）は、その曲線版です。

- 変数の重み（ベクトル）の代わりに、時刻ごとの重み（関数）$\xi_j(t)$ を求める。これを固有関数（eigenfunction）と呼ぶ。
- 曲線 $i$ の第 $j$ 主成分のスコア（score）は、$z_{ij} = \int \xi_j(t)\,\{x_i(t) - \hat{\mu}(t)\}\,dt$。
- スコアの分散 $\nu_j$ が、固有値（eigenvalue）にあたる。

固有関数の符号は、逆にしても同じ主成分です。ここでは、午前（12時より前）の重みの合計が正になる向きにそろえます。

In [ ]:
# 関数主成分分析（K 個の主成分）。固有関数の符号は、午前の重みの合計が正になる向きにそろえる
fpca <- function(fd, K) {
  p <- fda::pca.fd(fd, nharm = K)
  s <- sign(colSums(curve_values(p$harmonics)[tt < 12, , drop = FALSE]))
  p$harmonics$coefs <- sweep(p$harmonics$coefs, 2, s, "*")
  p$scores <- sweep(p$scores, 2, s, "*")
  colnames(p$scores) <- paste0("z", 1:K)
  p
}
# 新しい曲線のスコア（平均関数と固有関数は、p のものを使う）
fpca_scores <- function(fd_new, p) {
  centered <- fd_new
  centered$coefs <- fd_new$coefs - as.numeric(p$meanfd$coefs)
  s <- fda::inprod(centered, p$harmonics)
  colnames(s) <- colnames(p$scores)
  s
}

In [ ]:
# 図：平均関数 ± 2 × sqrt(固有値) × 固有関数
plot_weights <- function(p, js = 1:2, y_lab = "Value") {
  mu <- as.numeric(curve_values(p$meanfd))
  xi <- curve_values(p$harmonics)
  d <- do.call(rbind, lapply(js, function(j) {
    shift <- 2 * sqrt(p$values[j]) * xi[, j]
    data.frame(time = rep(tt, 3),
               component = sprintf("FPC %d (%.1f%%)", j, 100 * p$varprop[j]),
               kind = rep(c("mean", "mean + 2 sd", "mean - 2 sd"), each = 48),
               value = c(mu, mu + shift, mu - shift))
  }))
  ggplot(d, aes(time, value, colour = kind, linetype = kind)) +
    geom_line() +
    facet_wrap(~component) +
    scale_colour_manual(values = c("mean" = "black", "mean + 2 sd" = "#B2182B", "mean - 2 sd" = "#2166AC")) +
    scale_linetype_manual(values = c("mean" = "solid", "mean + 2 sd" = "dashed", "mean - 2 sd" = "dotted")) +
    scale_x_continuous(breaks = seq(0, 24, 6)) +
    labs(x = "Time of day (hour)", y = y_lab, colour = NULL, linetype = NULL)
}

In [ ]:
pca_sim <- fpca(fd_sim, K = 4)
# 固有値（スコアの分散）。本当の値は 6 と 2、3つ目からは 0
round(pca_sim$values[1:4], 3)
# 曲線を作るときに使った z1, z2 の、この200本での分散
round(c(var(z1), var(z2)), 3)
# 寄与率
round(pca_sim$varprop, 3)

固有値は、この200本での $z_1$、$z_2$ の分散とほぼ同じです。本当の値（6と2）との差は、200本しかないことによる標本の誤差です。

固有関数は、「平均関数 ± 定数 × 固有関数」の図で読みます。定数は、スコアの標準偏差の2倍（$2\sqrt{\nu_j}$）です。

In [ ]:
plot_weights(pca_sim, 1:2)

- 第1主成分：スコアが大きい曲線は、1日全体が高い。
- 第2主成分：スコアが大きい曲線は、午前が高く、午後が低い。

曲線を作るときに使った $\xi_1(t)$、$\xi_2(t)$ と比べます。

In [ ]:
xi_hat <- curve_values(pca_sim$harmonics)
ggplot(rbind(data.frame(time = tt, component = "FPC 1", estimated = xi_hat[, 1], truth = xi1),
             data.frame(time = tt, component = "FPC 2", estimated = xi_hat[, 2], truth = xi2)), aes(time)) +
  geom_line(aes(y = truth), linetype = "dashed") +
  geom_line(aes(y = estimated), colour = "red") +
  facet_wrap(~component) +
  scale_x_continuous(breaks = seq(0, 24, 6)) +
  labs(x = "Time of day (hour)", y = "Eigenfunction (red: estimated, dashed: truth)")
# スコア：推定した値と、曲線を作るときに使った値の相関
c(z1 = cor(pca_sim$scores[, 1], z1), z2 = cor(pca_sim$scores[, 2], z2))

スコアの散布図です。1つの点が、1本の曲線です。

In [ ]:
ggplot(data.frame(pca_sim$scores), aes(z1, z2)) +
  geom_point(alpha = 0.6) +
  labs(x = "Score of FPC 1", y = "Score of FPC 2")

主成分分析は、多くの変数を扱う方法でした。曲線を48個の変数（48個の時刻の値）とみて、ふつうの主成分分析をしても、ほぼ同じ結果になります。
共分散の行列の固有値に、時刻の間隔（0.5時間）をかけると、関数主成分分析の固有値と比べられます。

In [ ]:
round(eigen(G_hat)$values[1:4] * 0.5, 3)

曲線は、平均関数と、少数の主成分で組み立て直せます。

$$
x_i(t) \approx \hat{\mu}(t) + \sum_{j=1}^{K} z_{ij}\,\xi_j(t)
$$

In [ ]:
# K 個の主成分で組み立て直した曲線と、もとの（なめらかにした）曲線のずれ
rebuild <- function(p, K) {
  as.numeric(curve_values(p$meanfd)) + curve_values(p$harmonics)[, 1:K, drop = FALSE] %*% t(p$scores[, 1:K, drop = FALSE])
}
sapply(1:4, function(K) rmse(rebuild(pca_sim, K), X_sim))

2個の主成分で、ずれはノイズの残り程度まで小さくなります。200本の曲線の違いは、1本あたり2つの数で言い表せます。

### A-5. 関数回帰

曲線ごとに、数が1つずつ付いているとします（例：その日の最高気温）。この数 $y_i$ を、曲線 $x_i(t)$ で説明します。
これを関数回帰（functional regression）といいます。

$$
y_i = \alpha + \int \beta(t)\,\{x_i(t) - \mu(t)\}\,dt + e_i
$$

$\beta(t)$ は係数関数（coefficient function）で、「どの時刻の値が、どちら向きに効くか」を表します。

人工の応答を作ります。本当の係数関数は $\beta(t) = 0.5\,\xi_1(t) - 3\,\xi_2(t)$ です。午前が低く、午後が高い曲線ほど、$y$ が大きくなります。

In [ ]:
set.seed(2)
beta_true <- 0.5 * xi1 - 3 * xi2
y_sim <- 10 + 0.5 * z1 - 3 * z2 + rnorm(N, sd = 1)   # 積分を計算すると、この形になる
ggplot(data.frame(time = tt, beta = beta_true), aes(time, beta)) +
  geom_hline(yintercept = 0, colour = "grey60") +
  geom_line() +
  scale_x_continuous(breaks = seq(0, 24, 6)) +
  labs(x = "Time of day (hour)", y = "True beta(t)")

$\beta(t)$ は関数なので、そのままでは決めるべき数が多すぎます。そこで、主成分のスコアに回帰します（主成分回帰の曲線版）。

$$
y_i = \alpha + \sum_{j=1}^{K} b_j\,z_{ij} + e_i, \qquad \hat{\beta}(t) = \sum_{j=1}^{K} \hat{b}_j\,\xi_j(t)
$$

予測の良し悪しは、第2回と同じく、値を隠して比べます。最後の50本をテスト用に取り分け、残りの150本（訓練用）だけで平均関数、固有関数、回帰係数を決めます。

In [ ]:
# スコアへの回帰
fpc_regression <- function(y, scores) lm(y ~ ., data = data.frame(y = y, scores))
# 係数関数 beta(t) の、48個の時刻での値
beta_curve <- function(fit, p) as.numeric(curve_values(p$harmonics) %*% coef(fit)[-1])
# 訓練用の曲線で当てはめ、テスト用の曲線で誤差を出す
#   mean：訓練用の y の平均、level：曲線の1日の平均だけで回帰、fpc：K 個の主成分のスコアで回帰
evaluate <- function(fd, y, train, test, K) {
  p <- fpca(fd[train], K)
  fit <- fpc_regression(y[train], p$scores)
  level <- colMeans(curve_values(fd))
  fit_level <- lm(y ~ level, data = data.frame(y = y[train], level = level[train]))
  pred <- list(mean  = rep(mean(y[train]), length(test)),
               level = predict(fit_level, data.frame(level = level[test])),
               fpc   = predict(fit, data.frame(fpca_scores(fd[test], p))))
  data.frame(model = names(pred),
             RMSE = sapply(pred, rmse, truth = y[test]),
             MAE  = sapply(pred, mae, truth = y[test]), row.names = NULL)
}

In [ ]:
h <- 50
train <- 1:(N - h)
test  <- (N - h + 1):N
pca_tr <- fpca(fd_sim[train], K = 2)                 # 訓練用の曲線だけで、主成分を決める
fit <- fpc_regression(y_sim[train], pca_tr$scores)
round(coef(fit), 3)
ggplot(data.frame(time = tt, estimated = beta_curve(fit, pca_tr), truth = beta_true), aes(time)) +
  geom_hline(yintercept = 0, colour = "grey60") +
  geom_line(aes(y = truth), linetype = "dashed") +
  geom_line(aes(y = estimated), colour = "red") +
  scale_x_continuous(breaks = seq(0, 24, 6)) +
  labs(x = "Time of day (hour)", y = "beta(t) (red: estimated, dashed: truth)")

回帰係数は、本当の値（0.5 と −3）に近くなります。係数関数も、本当の $\beta(t)$ とほぼ重なります。

テスト用の50本で、予測の誤差を比べます。比べる相手（ベンチマーク）は2つです。

- `mean`：訓練用の $y$ の平均で当てる。曲線を使わない。
- `level`：曲線の1日の平均（1つの数）だけで回帰する。曲線の形を使わない。

In [ ]:
evaluate(fd_sim, y_sim, train, test, K = 2)
# 主成分の数 K を変える
data.frame(K = 1:6, RMSE = sapply(1:6, function(K) evaluate(fd_sim, y_sim, train, test, K)$RMSE[3]))

- `fpc`の誤差は、応答に入れたノイズの標準偏差（1）に近くなります。
- `level`は、`mean`とあまり変わりません。この応答は、曲線の高さではなく、形（午前と午後の対比）で決まるからです。
- $K = 1$ では足りず、$K = 2$ で誤差が下がり、3個目からは下がりません。

### A-6. 曲線どうしは独立か

ここまでの計算は、200本の曲線を、同じ分布からの独立な観測（iid）として扱っています。
曲線が「1日」なら、となりの日どうしは似ているかもしれません。

スコア $z_{i1}$ が、曲線の順 $i$ についてAR(1)（$\phi = 0.7$）で従属している場合を調べます。分散は、独立な場合と同じ6にします。
200本のスコアの平均を2000回求めて、その散らばりを比べます。

In [ ]:
set.seed(3)
phi <- 0.7
sd1 <- sqrt(nu_true[1])                      # スコアの標準偏差
mean_iid <- replicate(2000, mean(rnorm(N, sd = sd1)))
mean_ar  <- replicate(2000, mean(arima.sim(list(ar = phi), n = N, sd = sd1 * sqrt(1 - phi^2))))
c(sd_iid = sd(mean_iid), sd_ar1 = sd(mean_ar), ratio = sd(mean_ar) / sd(mean_iid),
  theory = sqrt((1 + phi) / (1 - phi)))
# 推定した平均関数（本当の平均関数 ＋ スコアの平均 × 固有関数）を、30回分ずつ描く
d <- rbind(data.frame(long_curves(mu_true + outer(xi1, mean_iid[1:30])), case = "independent curves"),
           data.frame(long_curves(mu_true + outer(xi1, mean_ar[1:30])), case = "dependent curves (AR(1), phi = 0.7)"))
ggplot(d, aes(time, value, group = curve)) +
  geom_line(alpha = 0.4) +
  facet_wrap(~case) +
  scale_x_continuous(breaks = seq(0, 24, 6)) +
  labs(x = "Time of day (hour)", y = "Estimated mean function")

- 曲線どうしが従属していると、推定した平均関数の散らばりが大きくなります。
- 標準偏差の比は、第1回で導いた $\sqrt{(1+\phi)/(1-\phi)}$ にほぼ一致します。200本あっても、独立な200本分の情報はありません。
- 曲線どうしが独立かどうかは、スコアを曲線の順に並べて、第1回のACFで確かめられます。

## B. 実データを解析する

`tsibbledata`パッケージの`vic_elec`を使います。オーストラリアのビクトリア州の電力需要で、第2回でも使いました。
第2回は1日ごとの合計にまとめました。今日は、1日を、30分ごとの48点の曲線として扱います。

B節は、A節で作った関数（`smooth_curves`、`fpca`など）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
season   <- "all"   # 季節："all"（1年じゅう）、"warm"（10〜3月）、"cool"（4〜9月）
day_type <- "all"   # 日の種類："all"（すべて）、"work"（平日）、"off"（土日と祝日）
response <- "max"   # 回帰の応答："max"（その日の最高気温）か "min"（最低気温）
K        <- 4       # 回帰に使う主成分の数：1〜6
h        <- 60      # テスト用に取り分ける曲線の数（最後の h 本）：20, 40, 60 のどれか

### B-1. データを確かめる

In [ ]:
glimpse(vic_elec)
interval(vic_elec)
range(vic_elec$Date)
# 1日あたりの行の数
table(table(vic_elec$Date))

- 1行は30分です。`Demand`が電力需要（MWh）、`Temperature`がメルボルンの気温（℃）です。
- 期間は2012年1月1日から2014年12月31日です。
- 1日は48行のはずですが、46行の日と50行の日が3日ずつあります。夏時間の切り替えの日です。

### B-2. 前処理

1日を1本の曲線にします。

1. 48行そろっている日だけを残す（夏時間の切り替えの6日を除く）。
2. 電力需要を、48行 × 日数の行列に並べる。単位はGWhにする。
3. 日ごとの表を作る：最高気温、最低気温、月、日の種類（平日か、土日・祝日か）。
4. 設定で選んだ季節と日の種類に絞る。

In [ ]:
half <- vic_elec |>
  as_tibble() |>
  group_by(Date) |>
  filter(n() == 48) |>                         # 1
  ungroup()
Y_all <- matrix(half$Demand / 1000, nrow = 48)  # 2：48行 × 日数（1列が1日）
day_all <- half |>
  group_by(Date) |>
  summarise(max = max(Temperature), min = min(Temperature), Holiday = any(Holiday)) |>   # 3
  mutate(month = as.integer(format(Date, "%m")),
         type = ifelse(Holiday | format(Date, "%u") %in% c("6", "7"), "off", "work"))
dim(Y_all)
months <- list(all = 1:12, warm = c(10:12, 1:3), cool = 4:9)[[season]]
keep <- which(day_all$month %in% months & (day_type == "all" | day_all$type == day_type))   # 4
Y <- Y_all[, keep]
day <- day_all[keep, ]
y <- day[[response]]
n <- ncol(Y)
n
table(day$type)

In [ ]:
show <- round(seq(1, n, length.out = 100))      # 100本だけ描く
plot_curves(Y, show, "Demand (GWh per 30 min)")

### B-3. なめらかな関数にする

In [ ]:
sel <- choose_lambda(Y)
lambda <- sel$lambda
log10(lambda)
fd <- smooth_curves(Y, lambda)$fd
X <- curve_values(fd)
plot_one(Y, fd, 1, sprintf("Demand (GWh per 30 min), %s", day$Date[1]))
# 観測となめらかにした曲線のずれと、曲線全体の散らばり
c(residual_sd = rmse(X, Y), overall_sd = sd(Y))

人工データと違って、観測の点は、はじめからほぼなめらかに並んでいます。ノイズが小さいので、選ばれる $\lambda$ も小さくなります。

### B-4. 平均関数と共分散関数

In [ ]:
mu_hat <- rowMeans(X)
plot_curves(X, show, "Demand (GWh per 30 min)") +
  geom_line(data = data.frame(time = tt, value = mu_hat, curve = 0), colour = "red", linewidth = 1.2)

In [ ]:
G_hat <- cov(t(X))
plot_cov(G_hat)
# 時刻ごとの標準偏差（4時、8時、16時ごろ）
round(sqrt(diag(G_hat))[c(8, 16, 32)], 3)

### B-5. 関数主成分分析

In [ ]:
pca <- fpca(fd, K = 6)
round(pca$varprop, 3)
round(cumsum(pca$varprop), 3)

In [ ]:
plot_weights(pca, 1:3, "Demand (GWh per 30 min)")

スコアの散布図を、日の種類と、その日の気温で色分けします。

In [ ]:
sc <- data.frame(pca$scores, type = day$type, temperature = y, curve = 1:n)
ggplot(sc, aes(z1, z2, colour = type)) +
  geom_point(alpha = 0.5, size = 1) +
  labs(x = "Score of FPC 1", y = "Score of FPC 2", colour = "Day type")
ggplot(sc, aes(z1, z2, colour = temperature)) +
  geom_point(alpha = 0.7, size = 1) +
  scale_colour_gradient2(low = "#2166AC", mid = "grey80", high = "#B2182B", midpoint = median(y)) +
  labs(x = "Score of FPC 1", y = "Score of FPC 2", colour = sprintf("Daily %s temp. (deg C)", response))

### B-6. 曲線どうしは独立か

スコアを日の順に並べて、ACFを見ます。日付は連続していないことがあるので、曲線の通し番号を時点にします（第1回の取引日と同じ扱いです）。

In [ ]:
sc_ts <- as_tsibble(sc, index = curve)
sc_ts |>
  ACF(z1, lag_max = 21) |>
  autoplot() +
  labs(x = "Lag (curves)", y = "Sample ACF of FPC 1 score")
acf_z1 <- ACF(sc_ts, z1, lag_max = 7)$acf
round(acf_z1, 2)

### B-7. 関数回帰

応答は、その日の気温です。1日の電力需要の曲線から、その日の気温を当てます。
最後の`h`本をテスト用に取り分け、残りの曲線だけで主成分と回帰係数を決めます。

In [ ]:
train <- 1:(n - h)
test  <- (n - h + 1):n
pca_tr <- fpca(fd[train], K)
fit <- fpc_regression(y[train], pca_tr$scores)
round(summary(fit)$coefficients, 3)
r2 <- summary(fit)$r.squared
r2
beta_hat <- beta_curve(fit, pca_tr)
ggplot(data.frame(time = tt, beta = beta_hat), aes(time, beta)) +
  geom_hline(yintercept = 0, colour = "grey60") +
  geom_line() +
  scale_x_continuous(breaks = seq(0, 24, 6)) +
  labs(x = "Time of day (hour)", y = "Estimated beta(t)")

In [ ]:
acc <- evaluate(fd, y, train, test, K)
acc
pred <- predict(fit, data.frame(fpca_scores(fd[test], pca_tr)))
ggplot(data.frame(observed = y[test], predicted = pred), aes(predicted, observed)) +
  geom_abline(colour = "grey60") +
  geom_point() +
  labs(x = "Predicted temperature (deg C)", y = "Observed temperature (deg C)")

### B-8. 診断と、くり返しの評価

回帰の残差を日の順に並べて、ACFを見ます。

In [ ]:
res_ts <- tsibble(curve = train, residual = as.numeric(residuals(fit)), index = curve)
res_ts |>
  ACF(residual, lag_max = 21) |>
  autoplot() +
  labs(x = "Lag (curves)", y = "Sample ACF of residuals")
acf_res <- ACF(res_ts, residual, lag_max = 7)$acf
round(acf_res, 2)

1回の分け方だけでは、たまたまその期間で当たっただけかもしれません。
第2回の時系列交差検証と同じように、訓練用の曲線を`h`本ずつ伸ばしながら、3回評価します（曲線が少なく、最初の訓練用が40本に満たなくなる設定では、2回に減らします）。

In [ ]:
n_rep <- min(3, (n - 40) %/% h)     # 評価の回数
folds <- lapply(n_rep:1, function(k) evaluate(fd, y, 1:(n - k * h), (n - k * h + 1):(n - (k - 1) * h), K))
acc_cv <- data.frame(model = folds[[1]]$model,
                     RMSE = sqrt(rowMeans(sapply(folds, function(f) f$RMSE^2))),
                     MAE  = rowMeans(sapply(folds, function(f) f$MAE)))
acc_cv

読み方：

- B-3：実データの曲線は、もとからなめらかです。なめらかにする効果は、人工データほど大きくありません。
- B-5：人工データのスコアは、1つのかたまりでした。実データのスコアは、いくつかの群に分かれます。群に分ける方法は、次回（第13回）に扱います。
- B-6：スコアのACFは、破線の外に出ます。曲線どうしは独立ではありません。A-6で見たとおり、平均関数や回帰係数の精度は、本数から期待するほど高くありません。
- B-7：`level`（1日の平均だけの回帰）が`mean`に負けることがあります。暑い日も寒い日も需要が増えるので、需要の高さだけでは気温が決まらないからです。曲線の形を使うと、当てられます。
- B-8：残差にも従属性が残ります。テスト用の曲線を最後から取り分けたのは、このためです。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `season` | `"all"` `"warm"`（10〜3月） `"cool"`（4〜9月） |
| `day_type` | `"all"` `"work"`（平日） `"off"`（土日と祝日） |
| `response` | `"max"`（最高気温） `"min"`（最低気温） |
| `K` | 1〜6 |
| `h` | 20、40、60 |

読むときの観点：

1. 季節を`"warm"`や`"cool"`に絞ると、主成分の形（B-5）と係数関数 $\beta(t)$（B-7）はどう変わるか。`level`の誤差は`mean`より小さくなるか。
2. 日の種類を`"work"`に絞ると、スコアの散布図の群はどうなるか。スコアのACF（B-6）のラグ7はどう変わるか。
3. 主成分の数`K`を1から6まで変えると、テストの誤差はどこまで下がるか。$\beta(t)$ の形は、`K`によってどう変わるか。
4. 応答を`"min"`にすると、効く時刻は変わるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（1年じゅう、すべての日、最高気温、K = 4、h = 60）の結果と比べてください。